# Phase 2 – Patient-Level Prediction Aggregation

**Goal**: combine all image-level predictions for a patient into a single patient-level decision, then compute sensitivity/specificity across patients rather than across images.

**What changes**: nothing is retrained. We post-process the saved OOF and test probabilities from Phase 1 (CE loss), Phase 2A (focal loss), and Phase 2B (focal + full fine-tune). The aggregation step runs in seconds.

**Two aggregation methods**:
- **Mean**: average each class's probability across all images → smooths noise, stable
- **Max**: highest probability seen per class across images → optimistic, better when signal is in one clear image

**Patient-level ground truth**: worst (highest) grade across all of a patient's images. This is consistent with how stratification was done for CV.

After aggregation, we test both argmax and Youden-threshold decisions.

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, accuracy_score, cohen_kappa_score, confusion_matrix
)

OUTPUT_DIR = Path('output_dir')
NUM_CLASSES = 4
CLASS_NAMES = ['R0', 'R1', 'R2', 'R3A']

## Load all saved predictions and metadata

In [2]:
# ── labels/splits.csv ───────────────────────────────────────────────
df_all  = pd.read_csv('labels/splits.csv')
df_cv   = df_all[df_all['split'].isin(['train','val'])].copy().reset_index(drop=True)
df_test = df_all[df_all['split'] == 'test'].copy().reset_index(drop=True)

# patient codes aligned with each array position
oof_codes  = df_cv['code'].values          # len 4075
test_codes = df_test['code'].values        # len 702

print(f'CV images: {len(df_cv)} | patients: {df_cv["code"].nunique()}')
print(f'Test images: {len(df_test)} | patients: {df_test["code"].nunique()}')

# ── Phase 1 ─────────────────────────────────────────────────────────
p1_oof_prb = np.load(OUTPUT_DIR/'phase1_cv/oof_probs_all.npy').astype(np.float64)
p1_oof_lbl = np.load(OUTPUT_DIR/'phase1_cv/oof_labels_all.npy').astype(int)
p1_tst_prb = np.load(OUTPUT_DIR/'phase1_cv/test_ensemble_probs.npy').astype(np.float64)
p1_tst_lbl = np.load(OUTPUT_DIR/'phase1_cv/test_ensemble_labels.npy').astype(int)

# ── Phase 2A ─────────────────────────────────────────────────────────
p2a_oof_prb = np.load(OUTPUT_DIR/'phase2a_cv/oof_probs_all.npy').astype(np.float64)
p2a_oof_lbl = np.load(OUTPUT_DIR/'phase2a_cv/oof_labels_all.npy').astype(int)
p2a_tst_prb = np.load(OUTPUT_DIR/'phase2a_cv/test_ensemble_probs.npy').astype(np.float64)
p2a_tst_lbl = np.load(OUTPUT_DIR/'phase2a_cv/test_ensemble_labels.npy').astype(int)

# ── Phase 2B (full fine-tune) ────────────────────────────────────────
p2b_oof_prb = np.load(OUTPUT_DIR/'phase2b_cv/oof_probs_all.npy').astype(np.float64)
p2b_oof_lbl = np.load(OUTPUT_DIR/'phase2b_cv/oof_labels_all.npy').astype(int)
p2b_tst_prb = np.load(OUTPUT_DIR/'phase2b_cv/test_ensemble_probs.npy').astype(np.float64)
p2b_tst_lbl = np.load(OUTPUT_DIR/'phase2b_cv/test_ensemble_labels.npy').astype(int)

# normalise rows to sum to 1 (eliminates float32 rounding residue)
def _norm(arr):
    return arr / arr.sum(axis=1, keepdims=True)

p1_oof_prb  = _norm(p1_oof_prb);  p1_tst_prb  = _norm(p1_tst_prb)
p2a_oof_prb = _norm(p2a_oof_prb); p2a_tst_prb = _norm(p2a_tst_prb)
p2b_oof_prb = _norm(p2b_oof_prb); p2b_tst_prb = _norm(p2b_tst_prb)

# ── Youden thresholds ────────────────────────────────────────────────
with open(OUTPUT_DIR/'phase2c_thresholds/thresholds.json') as f:
    thr1 = json.load(f)
p1_thresholds = [thr1['youden_thresholds'][c] for c in CLASS_NAMES]

with open(OUTPUT_DIR/'phase2a_cv/phase2a_summary.json') as f:
    thr2a = json.load(f)
p2a_thresholds = [thr2a['youden_thresholds_p2a'][c] for c in CLASS_NAMES]

with open(OUTPUT_DIR/'phase2b_cv/phase2b_summary.json') as f:
    thr2b = json.load(f)
p2b_thresholds = [thr2b['youden_thresholds_p2b'][c] for c in CLASS_NAMES]

print('\nPhase1  Youden thresholds:', [f'{t:.4f}' for t in p1_thresholds])
print('Phase2A Youden thresholds:', [f'{t:.4f}' for t in p2a_thresholds])
print('Phase2B Youden thresholds:', [f'{t:.4f}' for t in p2b_thresholds])

CV images: 4075 | patients: 990
Test images: 702 | patients: 175

Phase1  Youden thresholds: ['0.4907', '0.3160', '0.1252', '0.1463']
Phase2A Youden thresholds: ['0.3942', '0.3706', '0.1646', '0.1627']
Phase2B Youden thresholds: ['0.5049', '0.4509', '0.1035', '0.0435']


## Helper functions

In [3]:
def aggregate_by_patient(codes, probs, labels, method='mean'):
    """
    Group image-level predictions by patient code.

    Patient ground truth = worst (highest) grade across their images.
    Aggregation:
      method='mean' → average probability per class
      method='max'  → maximum probability per class, then renormalise

    Returns: patient_probs (N_patients, 4), patient_labels (N_patients,)
    """
    records = {}
    for code, prob, label in zip(codes, probs, labels):
        if code not in records:
            records[code] = {'probs': [], 'worst_grade': 0}
        records[code]['probs'].append(prob)
        records[code]['worst_grade'] = max(records[code]['worst_grade'], int(label))

    patient_codes  = sorted(records.keys())
    agg_probs  = []
    agg_labels = []

    for code in patient_codes:
        stack = np.stack(records[code]['probs'])   # (n_images, 4)
        if method == 'mean':
            p = stack.mean(axis=0)
        elif method == 'max':
            p = stack.max(axis=0)
            p = p / p.sum()    # renormalise so rows still sum to 1
        else:
            raise ValueError(f'Unknown method: {method}')
        agg_probs.append(p)
        agg_labels.append(records[code]['worst_grade'])

    return np.array(agg_probs), np.array(agg_labels)


def apply_thresholds(probs, thresholds):
    """
    OvR threshold decision rule:
    - if any class probability exceeds its threshold, pick the class with
      the highest ratio (prob / threshold)
    - otherwise fall back to argmax
    """
    thresholds = np.array(thresholds)
    ratios = probs / thresholds
    above  = probs > thresholds
    return np.where(above.any(axis=1), ratios.argmax(axis=1), probs.argmax(axis=1))


def compute_metrics(labels, probs, preds=None):
    if preds is None:
        preds = probs.argmax(axis=1)
    try:
        auroc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except Exception:
        auroc = float('nan')
    acc   = accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds, weights='quadratic')
    cm    = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for i in range(NUM_CLASSES):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp
        fp = cm[:,i].sum()-tp; tn = cm.sum()-tp-fn-fp
        sens.append(tp/(tp+fn) if (tp+fn)>0 else float('nan'))
        spec.append(tn/(tn+fp) if (tn+fp)>0 else float('nan'))
    return {
        'auroc': auroc, 'accuracy': acc, 'kappa': kappa,
        'macro_sens': np.nanmean(sens), 'macro_spec': np.nanmean(spec),
        'sensitivity': np.array(sens), 'specificity': np.array(spec),
    }


def print_metrics(label, m):
    print(f'\n── {label} ──')
    print(f'  AUROC={m["auroc"]:.4f}  Kappa={m["kappa"]:.4f}  Acc={m["accuracy"]:.4f}')
    print(f'  Macro sens={m["macro_sens"]:.4f}  Macro spec={m["macro_spec"]:.4f}')
    for i, (s, sp) in enumerate(zip(m['sensitivity'], m['specificity'])):
        print(f'  {CLASS_NAMES[i]}: sens={s:.4f}  spec={sp:.4f}')

print('Helpers ready.')

Helpers ready.


## Phase 1 OOF – image level vs patient level

In [4]:
# image-level baseline (existing behaviour)
m_p1_img_argmax  = compute_metrics(p1_oof_lbl, p1_oof_prb)
m_p1_img_youden  = compute_metrics(p1_oof_lbl, p1_oof_prb,
                                   preds=apply_thresholds(p1_oof_prb, p1_thresholds))

# patient-level aggregations
p1_pt_mean_prb, p1_pt_lbl = aggregate_by_patient(oof_codes, p1_oof_prb, p1_oof_lbl, 'mean')
p1_pt_max_prb,  _          = aggregate_by_patient(oof_codes, p1_oof_prb, p1_oof_lbl, 'max')

print(f'Patient-level OOF: {len(p1_pt_lbl)} patients')
vals, cnts = np.unique(p1_pt_lbl, return_counts=True)
print('  Class distribution:', dict(zip(vals.tolist(), cnts.tolist())))

m_p1_pt_mean_am  = compute_metrics(p1_pt_lbl, p1_pt_mean_prb)
m_p1_pt_mean_yo  = compute_metrics(p1_pt_lbl, p1_pt_mean_prb,
                                   preds=apply_thresholds(p1_pt_mean_prb, p1_thresholds))
m_p1_pt_max_am   = compute_metrics(p1_pt_lbl, p1_pt_max_prb)
m_p1_pt_max_yo   = compute_metrics(p1_pt_lbl, p1_pt_max_prb,
                                   preds=apply_thresholds(p1_pt_max_prb, p1_thresholds))

print_metrics('Phase1 OOF | Image-level + Argmax',  m_p1_img_argmax)
print_metrics('Phase1 OOF | Image-level + Youden',  m_p1_img_youden)
print_metrics('Phase1 OOF | Patient Mean + Argmax', m_p1_pt_mean_am)
print_metrics('Phase1 OOF | Patient Mean + Youden', m_p1_pt_mean_yo)
print_metrics('Phase1 OOF | Patient Max  + Argmax', m_p1_pt_max_am)
print_metrics('Phase1 OOF | Patient Max  + Youden', m_p1_pt_max_yo)

Patient-level OOF: 990 patients
  Class distribution: {0: 518, 1: 362, 2: 67, 3: 43}

── Phase1 OOF | Image-level + Argmax ──
  AUROC=0.8338  Kappa=0.6042  Acc=0.6746
  Macro sens=0.5756  Macro spec=0.8603
  R0: sens=0.8520  spec=0.6796
  R1: sens=0.4061  spec=0.8678
  R2: sens=0.5270  spec=0.9411
  R3A: sens=0.5172  spec=0.9527

── Phase1 OOF | Image-level + Youden ──
  AUROC=0.8338  Kappa=0.4724  Acc=0.5774
  Macro sens=0.5724  Macro spec=0.8480
  R0: sens=0.6803  spec=0.8067
  R1: sens=0.3815  spec=0.8079
  R2: sens=0.6763  spec=0.8704
  R3A: sens=0.5517  spec=0.9069

── Phase1 OOF | Patient Mean + Argmax ──
  AUROC=0.8712  Kappa=0.6926  Acc=0.7081
  Macro sens=0.6110  Macro spec=0.8707
  R0: sens=0.9517  spec=0.6271
  R1: sens=0.4116  spec=0.9283
  R2: sens=0.5224  spec=0.9621
  R3A: sens=0.5581  spec=0.9652

── Phase1 OOF | Patient Mean + Youden ──
  AUROC=0.8712  Kappa=0.5563  Acc=0.6152
  Macro sens=0.6106  Macro spec=0.8631
  R0: sens=0.7799  spec=0.8008
  R1: sens=0.3646  spec

## Phase 2A OOF – image level vs patient level

In [5]:
m_p2a_img_argmax  = compute_metrics(p2a_oof_lbl, p2a_oof_prb)
m_p2a_img_youden  = compute_metrics(p2a_oof_lbl, p2a_oof_prb,
                                    preds=apply_thresholds(p2a_oof_prb, p2a_thresholds))

p2a_pt_mean_prb, p2a_pt_lbl = aggregate_by_patient(oof_codes, p2a_oof_prb, p2a_oof_lbl, 'mean')
p2a_pt_max_prb,  _           = aggregate_by_patient(oof_codes, p2a_oof_prb, p2a_oof_lbl, 'max')

print(f'Patient-level OOF (Phase2A): {len(p2a_pt_lbl)} patients')
vals, cnts = np.unique(p2a_pt_lbl, return_counts=True)
print('  Class distribution:', dict(zip(vals.tolist(), cnts.tolist())))

m_p2a_pt_mean_am  = compute_metrics(p2a_pt_lbl, p2a_pt_mean_prb)
m_p2a_pt_mean_yo  = compute_metrics(p2a_pt_lbl, p2a_pt_mean_prb,
                                    preds=apply_thresholds(p2a_pt_mean_prb, p2a_thresholds))
m_p2a_pt_max_am   = compute_metrics(p2a_pt_lbl, p2a_pt_max_prb)
m_p2a_pt_max_yo   = compute_metrics(p2a_pt_lbl, p2a_pt_max_prb,
                                    preds=apply_thresholds(p2a_pt_max_prb, p2a_thresholds))

print_metrics('Phase2A OOF | Image-level + Argmax',  m_p2a_img_argmax)
print_metrics('Phase2A OOF | Image-level + Youden',  m_p2a_img_youden)
print_metrics('Phase2A OOF | Patient Mean + Argmax', m_p2a_pt_mean_am)
print_metrics('Phase2A OOF | Patient Mean + Youden', m_p2a_pt_mean_yo)
print_metrics('Phase2A OOF | Patient Max  + Argmax', m_p2a_pt_max_am)
print_metrics('Phase2A OOF | Patient Max  + Youden', m_p2a_pt_max_yo)

Patient-level OOF (Phase2A): 990 patients
  Class distribution: {0: 518, 1: 362, 2: 67, 3: 43}

── Phase2A OOF | Image-level + Argmax ──
  AUROC=0.8309  Kappa=0.6011  Acc=0.6763
  Macro sens=0.5806  Macro spec=0.8674
  R0: sens=0.7764  spec=0.7661
  R1: sens=0.5460  spec=0.7955
  R2: sens=0.5726  spec=0.9418
  R3A: sens=0.4276  spec=0.9662

── Phase2A OOF | Image-level + Youden ──
  AUROC=0.8309  Kappa=0.4917  Acc=0.5897
  Macro sens=0.5864  Macro spec=0.8521
  R0: sens=0.6964  spec=0.8067
  R1: sens=0.3807  spec=0.8218
  R2: sens=0.7510  spec=0.8581
  R3A: sens=0.5172  spec=0.9219

── Phase2A OOF | Patient Mean + Argmax ──
  AUROC=0.8648  Kappa=0.7248  Acc=0.7374
  Macro sens=0.6277  Macro spec=0.8867
  R0: sens=0.8861  spec=0.7394
  R1: sens=0.5856  spec=0.8599
  R2: sens=0.5970  spec=0.9653
  R3A: sens=0.4419  spec=0.9820

── Phase2A OOF | Patient Mean + Youden ──
  AUROC=0.8648  Kappa=0.6046  Acc=0.6515
  Macro sens=0.6380  Macro spec=0.8723
  R0: sens=0.8205  spec=0.7881
  R1: sen

## Phase 2B OOF – image level vs patient level

In [6]:
m_p2b_img_argmax  = compute_metrics(p2b_oof_lbl, p2b_oof_prb)
m_p2b_img_youden  = compute_metrics(p2b_oof_lbl, p2b_oof_prb,
                                    preds=apply_thresholds(p2b_oof_prb, p2b_thresholds))

p2b_pt_mean_prb, p2b_pt_lbl = aggregate_by_patient(oof_codes, p2b_oof_prb, p2b_oof_lbl, 'mean')
p2b_pt_max_prb,  _           = aggregate_by_patient(oof_codes, p2b_oof_prb, p2b_oof_lbl, 'max')

print(f'Patient-level OOF (Phase2B): {len(p2b_pt_lbl)} patients')
vals, cnts = np.unique(p2b_pt_lbl, return_counts=True)
print('  Class distribution:', dict(zip(vals.tolist(), cnts.tolist())))

m_p2b_pt_mean_am  = compute_metrics(p2b_pt_lbl, p2b_pt_mean_prb)
m_p2b_pt_mean_yo  = compute_metrics(p2b_pt_lbl, p2b_pt_mean_prb,
                                    preds=apply_thresholds(p2b_pt_mean_prb, p2b_thresholds))
m_p2b_pt_max_am   = compute_metrics(p2b_pt_lbl, p2b_pt_max_prb)
m_p2b_pt_max_yo   = compute_metrics(p2b_pt_lbl, p2b_pt_max_prb,
                                    preds=apply_thresholds(p2b_pt_max_prb, p2b_thresholds))

print_metrics('Phase2B OOF | Image-level + Argmax',  m_p2b_img_argmax)
print_metrics('Phase2B OOF | Image-level + Youden',  m_p2b_img_youden)
print_metrics('Phase2B OOF | Patient Mean + Argmax', m_p2b_pt_mean_am)
print_metrics('Phase2B OOF | Patient Mean + Youden', m_p2b_pt_mean_yo)
print_metrics('Phase2B OOF | Patient Max  + Argmax', m_p2b_pt_max_am)
print_metrics('Phase2B OOF | Patient Max  + Youden', m_p2b_pt_max_yo)

Patient-level OOF (Phase2B): 990 patients
  Class distribution: {0: 518, 1: 362, 2: 67, 3: 43}

── Phase2B OOF | Image-level + Argmax ──
  AUROC=0.9081  Kappa=0.7747  Acc=0.8106
  Macro sens=0.6844  Macro spec=0.9127
  R0: sens=0.9239  spec=0.7829
  R1: sens=0.6739  spec=0.8996
  R2: sens=0.6639  spec=0.9755
  R3A: sens=0.4759  spec=0.9926

── Phase2B OOF | Image-level + Youden ──
  AUROC=0.9081  Kappa=0.6533  Acc=0.7379
  Macro sens=0.6872  Macro spec=0.8989
  R0: sens=0.8801  spec=0.8137
  R1: sens=0.4996  spec=0.9120
  R2: sens=0.7344  spec=0.9379
  R3A: sens=0.6345  spec=0.9318

── Phase2B OOF | Patient Mean + Argmax ──
  AUROC=0.9166  Kappa=0.7726  Acc=0.8030
  Macro sens=0.6575  Macro spec=0.9107
  R0: sens=0.9498  spec=0.7669
  R1: sens=0.6796  spec=0.8997
  R2: sens=0.5821  spec=0.9805
  R3A: sens=0.4186  spec=0.9958

── Phase2B OOF | Patient Mean + Youden ──
  AUROC=0.9166  Kappa=0.6570  Acc=0.7384
  Macro sens=0.7046  Macro spec=0.9045
  R0: sens=0.8977  spec=0.8263
  R1: sen

## Test set: Phase 1

In [7]:
m_p1_tst_img_am  = compute_metrics(p1_tst_lbl, p1_tst_prb)
m_p1_tst_img_yo  = compute_metrics(p1_tst_lbl, p1_tst_prb,
                                   preds=apply_thresholds(p1_tst_prb, p1_thresholds))

p1_tst_pt_mean_prb, p1_tst_pt_lbl = aggregate_by_patient(test_codes, p1_tst_prb, p1_tst_lbl, 'mean')
p1_tst_pt_max_prb,  _              = aggregate_by_patient(test_codes, p1_tst_prb, p1_tst_lbl, 'max')

print(f'Patient-level Test (Phase1): {len(p1_tst_pt_lbl)} patients')
vals, cnts = np.unique(p1_tst_pt_lbl, return_counts=True)
print('  Class distribution:', dict(zip(vals.tolist(), cnts.tolist())))

m_p1_tst_pt_mean_am  = compute_metrics(p1_tst_pt_lbl, p1_tst_pt_mean_prb)
m_p1_tst_pt_mean_yo  = compute_metrics(p1_tst_pt_lbl, p1_tst_pt_mean_prb,
                                       preds=apply_thresholds(p1_tst_pt_mean_prb, p1_thresholds))
m_p1_tst_pt_max_am   = compute_metrics(p1_tst_pt_lbl, p1_tst_pt_max_prb)
m_p1_tst_pt_max_yo   = compute_metrics(p1_tst_pt_lbl, p1_tst_pt_max_prb,
                                       preds=apply_thresholds(p1_tst_pt_max_prb, p1_thresholds))

print_metrics('Phase1 Test | Image-level + Argmax',  m_p1_tst_img_am)
print_metrics('Phase1 Test | Image-level + Youden',  m_p1_tst_img_yo)
print_metrics('Phase1 Test | Patient Mean + Argmax', m_p1_tst_pt_mean_am)
print_metrics('Phase1 Test | Patient Mean + Youden', m_p1_tst_pt_mean_yo)
print_metrics('Phase1 Test | Patient Max  + Argmax', m_p1_tst_pt_max_am)
print_metrics('Phase1 Test | Patient Max  + Youden', m_p1_tst_pt_max_yo)

Patient-level Test (Phase1): 175 patients
  Class distribution: {0: 91, 1: 63, 2: 12, 3: 9}



── Phase1 Test | Image-level + Argmax ──
  AUROC=0.8259  Kappa=0.6665  Acc=0.6866
  Macro sens=0.5554  Macro spec=0.8669
  R0: sens=0.9054  spec=0.6945
  R1: sens=0.3898  spec=0.8734
  R2: sens=0.5098  spec=0.9278
  R3A: sens=0.4167  spec=0.9720

── Phase1 Test | Image-level + Youden ──
  AUROC=0.8259  Kappa=0.5421  Acc=0.5969
  Macro sens=0.5591  Macro spec=0.8558
  R0: sens=0.7340  spec=0.8199
  R1: sens=0.3602  spec=0.8219
  R2: sens=0.7255  spec=0.8387
  R3A: sens=0.4167  spec=0.9425

── Phase1 Test | Patient Mean + Argmax ──
  AUROC=0.8705  Kappa=0.7257  Acc=0.7086
  Macro sens=0.5823  Macro spec=0.8737
  R0: sens=0.9560  spec=0.6667
  R1: sens=0.4286  spec=0.9018
  R2: sens=0.5000  spec=0.9325
  R3A: sens=0.4444  spec=0.9940

── Phase1 Test | Patient Mean + Youden ──
  AUROC=0.8705  Kappa=0.6078  Acc=0.6400
  Macro sens=0.5736  Macro spec=0.8757
  R0: sens=0.8462  spec=0.8333
  R1: sens=0.3651  spec=0.9018
  R2: sens=0.7500  spec=0.8221
  R3A: sens=0.3333  spec=0.9458

── Phase1

## Test set: Phase 2A

In [8]:
m_p2a_tst_img_am  = compute_metrics(p2a_tst_lbl, p2a_tst_prb)
m_p2a_tst_img_yo  = compute_metrics(p2a_tst_lbl, p2a_tst_prb,
                                    preds=apply_thresholds(p2a_tst_prb, p2a_thresholds))

p2a_tst_pt_mean_prb, p2a_tst_pt_lbl = aggregate_by_patient(test_codes, p2a_tst_prb, p2a_tst_lbl, 'mean')
p2a_tst_pt_max_prb,  _               = aggregate_by_patient(test_codes, p2a_tst_prb, p2a_tst_lbl, 'max')

print(f'Patient-level Test (Phase2A): {len(p2a_tst_pt_lbl)} patients')
vals, cnts = np.unique(p2a_tst_pt_lbl, return_counts=True)
print('  Class distribution:', dict(zip(vals.tolist(), cnts.tolist())))

m_p2a_tst_pt_mean_am  = compute_metrics(p2a_tst_pt_lbl, p2a_tst_pt_mean_prb)
m_p2a_tst_pt_mean_yo  = compute_metrics(p2a_tst_pt_lbl, p2a_tst_pt_mean_prb,
                                        preds=apply_thresholds(p2a_tst_pt_mean_prb, p2a_thresholds))
m_p2a_tst_pt_max_am   = compute_metrics(p2a_tst_pt_lbl, p2a_tst_pt_max_prb)
m_p2a_tst_pt_max_yo   = compute_metrics(p2a_tst_pt_lbl, p2a_tst_pt_max_prb,
                                        preds=apply_thresholds(p2a_tst_pt_max_prb, p2a_thresholds))

print_metrics('Phase2A Test | Image-level + Argmax',  m_p2a_tst_img_am)
print_metrics('Phase2A Test | Image-level + Youden',  m_p2a_tst_img_yo)
print_metrics('Phase2A Test | Patient Mean + Argmax', m_p2a_tst_pt_mean_am)
print_metrics('Phase2A Test | Patient Mean + Youden', m_p2a_tst_pt_mean_yo)
print_metrics('Phase2A Test | Patient Max  + Argmax', m_p2a_tst_pt_max_am)
print_metrics('Phase2A Test | Patient Max  + Youden', m_p2a_tst_pt_max_yo)

Patient-level Test (Phase2A): 175 patients
  Class distribution: {0: 91, 1: 63, 2: 12, 3: 9}

── Phase2A Test | Image-level + Argmax ──
  AUROC=0.8180  Kappa=0.6491  Acc=0.6838
  Macro sens=0.5361  Macro spec=0.8718
  R0: sens=0.8389  spec=0.7717
  R1: sens=0.5042  spec=0.8112
  R2: sens=0.5098  spec=0.9324
  R3A: sens=0.2917  spec=0.9720

── Phase2A Test | Image-level + Youden ──
  AUROC=0.8180  Kappa=0.5822  Acc=0.6211
  Macro sens=0.5632  Macro spec=0.8621
  R0: sens=0.7852  spec=0.8039
  R1: sens=0.3475  spec=0.8476
  R2: sens=0.7451  spec=0.8618
  R3A: sens=0.3750  spec=0.9351

── Phase2A Test | Patient Mean + Argmax ──
  AUROC=0.8595  Kappa=0.7695  Acc=0.7829
  Macro sens=0.6281  Macro spec=0.9096
  R0: sens=0.9451  spec=0.8095
  R1: sens=0.6508  spec=0.8839
  R2: sens=0.5833  spec=0.9509
  R3A: sens=0.3333  spec=0.9940

── Phase2A Test | Patient Mean + Youden ──
  AUROC=0.8595  Kappa=0.6408  Acc=0.6457
  Macro sens=0.5559  Macro spec=0.8758
  R0: sens=0.8901  spec=0.8095
  R1: s

## Test set: Phase 2B

In [9]:
m_p2b_tst_img_am  = compute_metrics(p2b_tst_lbl, p2b_tst_prb)
m_p2b_tst_img_yo  = compute_metrics(p2b_tst_lbl, p2b_tst_prb,
                                    preds=apply_thresholds(p2b_tst_prb, p2b_thresholds))

p2b_tst_pt_mean_prb, p2b_tst_pt_lbl = aggregate_by_patient(test_codes, p2b_tst_prb, p2b_tst_lbl, 'mean')
p2b_tst_pt_max_prb,  _               = aggregate_by_patient(test_codes, p2b_tst_prb, p2b_tst_lbl, 'max')

print(f'Patient-level Test (Phase2B): {len(p2b_tst_pt_lbl)} patients')
vals, cnts = np.unique(p2b_tst_pt_lbl, return_counts=True)
print('  Class distribution:', dict(zip(vals.tolist(), cnts.tolist())))

m_p2b_tst_pt_mean_am  = compute_metrics(p2b_tst_pt_lbl, p2b_tst_pt_mean_prb)
m_p2b_tst_pt_mean_yo  = compute_metrics(p2b_tst_pt_lbl, p2b_tst_pt_mean_prb,
                                        preds=apply_thresholds(p2b_tst_pt_mean_prb, p2b_thresholds))
m_p2b_tst_pt_max_am   = compute_metrics(p2b_tst_pt_lbl, p2b_tst_pt_max_prb)
m_p2b_tst_pt_max_yo   = compute_metrics(p2b_tst_pt_lbl, p2b_tst_pt_max_prb,
                                        preds=apply_thresholds(p2b_tst_pt_max_prb, p2b_thresholds))

print_metrics('Phase2B Test | Image-level + Argmax',  m_p2b_tst_img_am)
print_metrics('Phase2B Test | Image-level + Youden',  m_p2b_tst_img_yo)
print_metrics('Phase2B Test | Patient Mean + Argmax', m_p2b_tst_pt_mean_am)
print_metrics('Phase2B Test | Patient Mean + Youden', m_p2b_tst_pt_mean_yo)
print_metrics('Phase2B Test | Patient Max  + Argmax', m_p2b_tst_pt_max_am)
print_metrics('Phase2B Test | Patient Max  + Youden', m_p2b_tst_pt_max_yo)

Patient-level Test (Phase2B): 175 patients
  Class distribution: {0: 91, 1: 63, 2: 12, 3: 9}

── Phase2B Test | Image-level + Argmax ──
  AUROC=0.9271  Kappa=0.7671  Acc=0.8305
  Macro sens=0.6375  Macro spec=0.9259
  R0: sens=0.9463  spec=0.8424
  R1: sens=0.7458  spec=0.8884
  R2: sens=0.6078  spec=0.9800
  R3A: sens=0.2500  spec=0.9926

── Phase2B Test | Image-level + Youden ──
  AUROC=0.9271  Kappa=0.7061  Acc=0.7450
  Macro sens=0.6462  Macro spec=0.9070
  R0: sens=0.9079  spec=0.8521
  R1: sens=0.5127  spec=0.9227
  R2: sens=0.7059  spec=0.9017
  R3A: sens=0.4583  spec=0.9513

── Phase2B Test | Patient Mean + Argmax ──
  AUROC=0.9456  Kappa=0.8212  Acc=0.8629
  Macro sens=0.6788  Macro spec=0.9389
  R0: sens=0.9890  spec=0.8571
  R1: sens=0.8095  spec=0.9107
  R2: sens=0.5833  spec=0.9877
  R3A: sens=0.3333  spec=1.0000

── Phase2B Test | Patient Mean + Youden ──
  AUROC=0.9456  Kappa=0.7681  Acc=0.7543
  Macro sens=0.6634  Macro spec=0.9163
  R0: sens=0.9670  spec=0.8690
  R1: s

## Master Summary – Test Set (all phases × all decision rules)

In [10]:
rows = [
    ('P1  | Image  | Argmax', m_p1_tst_img_am),
    ('P1  | Image  | Youden', m_p1_tst_img_yo),
    ('P1  | Pt Mean| Argmax', m_p1_tst_pt_mean_am),
    ('P1  | Pt Mean| Youden', m_p1_tst_pt_mean_yo),
    ('P1  | Pt Max | Argmax', m_p1_tst_pt_max_am),
    ('P1  | Pt Max | Youden', m_p1_tst_pt_max_yo),
    ('P2A | Image  | Argmax', m_p2a_tst_img_am),
    ('P2A | Image  | Youden', m_p2a_tst_img_yo),
    ('P2A | Pt Mean| Argmax', m_p2a_tst_pt_mean_am),
    ('P2A | Pt Mean| Youden', m_p2a_tst_pt_mean_yo),
    ('P2A | Pt Max | Argmax', m_p2a_tst_pt_max_am),
    ('P2A | Pt Max | Youden', m_p2a_tst_pt_max_yo),
    ('P2B | Image  | Argmax', m_p2b_tst_img_am),
    ('P2B | Image  | Youden', m_p2b_tst_img_yo),
    ('P2B | Pt Mean| Argmax', m_p2b_tst_pt_mean_am),
    ('P2B | Pt Mean| Youden', m_p2b_tst_pt_mean_yo),
    ('P2B | Pt Max | Argmax', m_p2b_tst_pt_max_am),
    ('P2B | Pt Max | Youden', m_p2b_tst_pt_max_yo),
]

cols = ['Configuration', 'AUROC', 'Kappa', 'MacroSens', 'MacroSpec', 'R0 Sens', 'R1 Sens', 'R2 Sens', 'R3A Sens']
header = f"{cols[0]:<26} | {cols[1]:>6} | {cols[2]:>6} | {cols[3]:>9} | {cols[4]:>9} | {cols[5]:>7} | {cols[6]:>7} | {cols[7]:>7} | {cols[8]:>8}"
print(header)
print('-' * len(header))

prev_phase = None
for label, m in rows:
    phase = label[:3]
    if prev_phase and phase != prev_phase:
        print()
    prev_phase = phase
    s = m['sensitivity']
    print(
        f"{label:<26} | {m['auroc']:>6.4f} | {m['kappa']:>6.4f} | "
        f"{m['macro_sens']:>9.4f} | {m['macro_spec']:>9.4f} | "
        f"{s[0]:>7.4f} | {s[1]:>7.4f} | {s[2]:>7.4f} | {s[3]:>8.4f}"
    )


Configuration              |  AUROC |  Kappa | MacroSens | MacroSpec | R0 Sens | R1 Sens | R2 Sens | R3A Sens
-------------------------------------------------------------------------------------------------------------
P1  | Image  | Argmax      | 0.8259 | 0.6665 |    0.5554 |    0.8669 |  0.9054 |  0.3898 |  0.5098 |   0.4167
P1  | Image  | Youden      | 0.8259 | 0.5421 |    0.5591 |    0.8558 |  0.7340 |  0.3602 |  0.7255 |   0.4167
P1  | Pt Mean| Argmax      | 0.8705 | 0.7257 |    0.5823 |    0.8737 |  0.9560 |  0.4286 |  0.5000 |   0.4444
P1  | Pt Mean| Youden      | 0.8705 | 0.6078 |    0.5736 |    0.8757 |  0.8462 |  0.3651 |  0.7500 |   0.3333
P1  | Pt Max | Argmax      | 0.8552 | 0.7211 |    0.5820 |    0.8625 |  0.9670 |  0.3333 |  0.5833 |   0.4444
P1  | Pt Max | Youden      | 0.8552 | 0.5612 |    0.5618 |    0.8513 |  0.7473 |  0.2222 |  0.8333 |   0.4444

P2A | Image  | Argmax      | 0.8180 | 0.6491 |    0.5361 |    0.8718 |  0.8389 |  0.5042 |  0.5098 |   0.2917
P2A | Ima

## R3A Sensitivity — Threshold Sweep (P2B OOF)

Keep R0/R1/R2 at their Youden values. Sweep only the R3A threshold to find the best R3A recall without collapsing other classes.

In [11]:
# Fix R0/R1/R2 at their P2B Youden values; sweep only R3A threshold.
# For each candidate threshold we check: how many of the 43 OOF R3A patients
# get correctly classified, and what it costs the other classes.

import numpy as np

r3a_candidates = np.concatenate([
    np.linspace(0.001, 0.04, 40),   # below current Youden (0.0435)
    np.linspace(0.04,  0.15, 40),   # above current Youden
])

base_thr = np.array(p2b_thresholds)   # R0=0.5049, R1=0.4509, R2=0.1035, R3A=0.0435

sweep_rows = []
for r3a_thr in r3a_candidates:
    thr = base_thr.copy()
    thr[3] = r3a_thr

    # --- image level ---
    img_preds = apply_thresholds(p2b_oof_prb, thr)
    img_m     = compute_metrics(p2b_oof_lbl, p2b_oof_prb, preds=img_preds)

    # --- patient level (mean pooling) ---
    pt_preds = apply_thresholds(p2b_pt_mean_prb, thr)
    pt_m     = compute_metrics(p2b_pt_lbl, p2b_pt_mean_prb, preds=pt_preds)

    sweep_rows.append({
        'r3a_thr': r3a_thr,
        'img_r3a':  img_m['sensitivity'][3],
        'img_r1':   img_m['sensitivity'][1],
        'img_kappa':img_m['kappa'],
        'pt_r3a':   pt_m['sensitivity'][3],
        'pt_r1':    pt_m['sensitivity'][1],
        'pt_r2':    pt_m['sensitivity'][2],
        'pt_kappa': pt_m['kappa'],
        'pt_macro': pt_m['macro_sens'],
    })

# Print a table at 10 evenly-spaced checkpoints + wherever R3A sens jumps
print(f"{'R3A Thr':>8} | {'Img R3A':>7} | {'Img R1':>6} | {'Img Kap':>7}"
      f" | {'Pt R3A':>6} | {'Pt R1':>6} | {'Pt R2':>6} | {'Pt Kap':>6} | {'Pt Macro':>8}")
print('-' * 80)

seen_r3a_pt = set()
for r in sweep_rows:
    bucket = round(r['pt_r3a'] * 10) / 10      # group by 0.1 increments
    if bucket not in seen_r3a_pt:
        seen_r3a_pt.add(bucket)
        print(f"{r['r3a_thr']:>8.4f} | {r['img_r3a']:>7.4f} | {r['img_r1']:>6.4f} | {r['img_kappa']:>7.4f}"
              f" | {r['pt_r3a']:>6.4f} | {r['pt_r1']:>6.4f} | {r['pt_r2']:>6.4f} | {r['pt_kappa']:>6.4f} | {r['pt_macro']:>8.4f}")


 R3A Thr | Img R3A | Img R1 | Img Kap | Pt R3A |  Pt R1 |  Pt R2 | Pt Kap | Pt Macro
--------------------------------------------------------------------------------
  0.0010 |  1.0000 | 0.0000 |  0.0000 | 1.0000 | 0.0000 | 0.0000 | 0.0000 |   0.2500
  0.0080 |  0.9172 | 0.2214 |  0.2902 | 0.9302 | 0.2044 | 0.0448 | 0.2843 |   0.4556
  0.0150 |  0.8345 | 0.3373 |  0.4812 | 0.8372 | 0.3425 | 0.1791 | 0.4816 |   0.5468
  0.0310 |  0.6828 | 0.4637 |  0.5995 | 0.7442 | 0.4807 | 0.6716 | 0.6230 |   0.6971
  0.0428 |  0.6345 | 0.4981 |  0.6516 | 0.6279 | 0.5138 | 0.7761 | 0.6558 |   0.7039
  0.0795 |  0.5310 | 0.5393 |  0.6971 | 0.5349 | 0.5691 | 0.8806 | 0.7222 |   0.7239


## Apply Chosen R3A Threshold — Test Set

In [12]:
# Pick the R3A threshold that gives the best R3A recall on OOF
# without dropping patient-level Kappa below 0.75 or R1 below 0.70.
# Inspect the sweep table above and set CHOSEN_R3A_THR accordingly.

CHOSEN_R3A_THR = None   # <-- fill in after reading the sweep table

if CHOSEN_R3A_THR is None:
    # Auto-select: highest pt R3A sens where pt Kappa >= 0.75 AND pt R1 >= 0.70
    best = None
    for r in sweep_rows:
        if r['pt_kappa'] >= 0.75 and r['pt_r1'] >= 0.70:
            if best is None or r['pt_r3a'] > best['pt_r3a']:
                best = r
    if best is None:
        print("No threshold meets both constraints — relaxing to Kappa >= 0.72")
        for r in sweep_rows:
            if r['pt_kappa'] >= 0.72:
                if best is None or r['pt_r3a'] > best['pt_r3a']:
                    best = r
    CHOSEN_R3A_THR = best['pt_r3a'] and best['r3a_thr']
    print(f"Auto-selected R3A threshold: {CHOSEN_R3A_THR:.4f}")
    print(f"  OOF patient: R3A={best['pt_r3a']:.4f}  R1={best['pt_r1']:.4f}  Kappa={best['pt_kappa']:.4f}")

# ── Apply to test set ────────────────────────────────────────────────
chosen_thr = np.array(p2b_thresholds)
chosen_thr[3] = CHOSEN_R3A_THR

# Image level
img_preds_new = apply_thresholds(p2b_tst_prb, chosen_thr)
m_img_new = compute_metrics(p2b_tst_lbl, p2b_tst_prb, preds=img_preds_new)

# Patient mean level
pt_preds_new = apply_thresholds(p2b_tst_pt_mean_prb, chosen_thr)
m_pt_new = compute_metrics(p2b_tst_pt_lbl, p2b_tst_pt_mean_prb, preds=pt_preds_new)

print(f"\nR3A threshold changed: {p2b_thresholds[3]:.4f} → {CHOSEN_R3A_THR:.4f}")
print()
print_metrics(f"P2B | Image   | R3A-tuned (thr={CHOSEN_R3A_THR:.4f})", m_img_new)
print()
print_metrics(f"P2B | Pt Mean | R3A-tuned (thr={CHOSEN_R3A_THR:.4f})", m_pt_new)
print()

# Compare against previous best
print("── Comparison on test set ──")
cols = ["Configuration", "AUROC", "Kappa", "R0", "R1", "R2", "R3A"]
hdr  = f"{cols[0]:<38} | {cols[1]:>6} | {cols[2]:>6} | {cols[3]:>6} | {cols[4]:>6} | {cols[5]:>6} | {cols[6]:>6}"
print(hdr)
print("-" * len(hdr))
for label, m in [
    ("P2B | Image  | Argmax (baseline)",  m_p2b_tst_img_am),
    ("P2B | Image  | Youden",             m_p2b_tst_img_yo),
    (f"P2B | Image  | R3A-tuned",         m_img_new),
    ("P2B | PtMean | Argmax (best so far)",m_p2b_tst_pt_mean_am),
    ("P2B | PtMean | Youden",             m_p2b_tst_pt_mean_yo),
    (f"P2B | PtMean | R3A-tuned",         m_pt_new),
]:
    s = m["sensitivity"]
    print(f"{label:<38} | {m['auroc']:>6.4f} | {m['kappa']:>6.4f} | {s[0]:>6.4f} | {s[1]:>6.4f} | {s[2]:>6.4f} | {s[3]:>6.4f}")


No threshold meets both constraints — relaxing to Kappa >= 0.72
Auto-selected R3A threshold: 0.0767
  OOF patient: R3A=0.5581  R1=0.5663  Kappa=0.7229

R3A threshold changed: 0.0435 → 0.0767


── P2B | Image   | R3A-tuned (thr=0.0767) ──
  AUROC=0.9271  Kappa=0.7603  Acc=0.7607
  Macro sens=0.6524  Macro spec=0.9108
  R0: sens=0.9207  spec=0.8521
  R1: sens=0.5297  spec=0.9206
  R2: sens=0.7843  spec=0.8955
  R3A: sens=0.3750  spec=0.9749


── P2B | Pt Mean | R3A-tuned (thr=0.0767) ──
  AUROC=0.9456  Kappa=0.7719  Acc=0.7600
  Macro sens=0.6604  Macro spec=0.9178
  R0: sens=0.9670  spec=0.8690
  R1: sens=0.5079  spec=0.9732
  R2: sens=0.8333  spec=0.8650
  R3A: sens=0.3333  spec=0.9639

── Comparison on test set ──
Configuration                          |  AUROC |  Kappa |     R0 |     R1 |     R2 |    R3A
--------------------------------------------------------------------------------------------
P2B | Image  | Argmax (baseline)       | 0.9271 | 0.7671 | 0.9463 | 0.7458 | 0.6078 | 0.2